# Assignment 1: Build, Train, and Evaluate Your First Neural Network

**Unit:** Foundations of Deep Learning | **Companion reading:** Lessons 1–3  
**Suggested time:** 1.5–2.5 hours

---

## What is this assignment trying to do?

You have likely heard about deep neural networks and image classifiers. This assignment shows **what happens underneath** when a model learns to recognize complex patterns from raw visual pixels.

In five steps you will:

1. **Load and inspect** a real dataset of handwritten digits (MNIST).
2. **Split** data into training, validation, and test sets to avoid data leakage and overfitting.
3. **Build** a Multi-Layer Perceptron (MLP) architecture using PyTorch.
4. **Train** your network using gradient descent and track performance over time.
5. **Evaluate** your final model and reflect on hyperparameter choices.

**Big idea:** Neural networks are collections of layers that transform raw inputs into target predictions by iteratively updating weights to minimize loss.

| Part | What you do | Why |
|------|-------------|-----|
| 1 | Load and visualize MNIST | Inspect your input data before training |
| 2 | Split data & set hyperparameters | Practice proper dataset split strategy |
| 3 | Complete PyTorch MLP architecture | Understand shape transformations and linear layers |
| 4 | Run training loop & compare experiments | Observe loss decrease & detect overfitting |
| 5 | Evaluate on test set & reflect | Measure true generalization performance |


---
## Key Terms (Read this first)

| Term | Plain-English Meaning |
|------|------------------------|
| **MNIST** | Benchmark dataset of 70,000 grayscale images ($28 \times 28$ pixels) of handwritten digits (0–9). |
| **Tensor** | PyTorch's primary data structure (multi-dimensional array) optimized for GPU operations. |
| **Flattening** | Unrolling a 2D image matrix ($28 \times 28$) into a 1D vector (784 numbers). |
| **Batch Size** | Number of image examples processed before updating the model's weights once. |
| **Learning Rate** | Step size multiplier used by gradient descent when updating weights. |
| **Epoch** | One complete pass through every training example in the dataset. |
| **Validation Set** | Data set aside during development to evaluate hyperparameters without cheating on the test set. |
| **Overfitting** | When a model learns training data noise and performs poorly on unseen validation data. |
| **Cross Entropy Loss** | Measure of error for multi-class classification tasks. **Lower is better.** |

**Analogy:** Training set = homework practice problems. Validation set = mid-term practice exam used to change study habits. Test set = final exam taken once at the end.


---
## Part 0: Environment Setup

Run the cell below to load PyTorch, Torchvision, Matplotlib, and set up your execution device (CPU vs GPU).

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


---
## Part 1: Download & Inspect Data

Before building models, machine learning engineers spend time understanding raw data format and image scaling.

In [ ]:
# Data Preprocessing: Convert PIL images to Tensors and normalize pixel intensities
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print(f"Raw training examples: {len(full_dataset)}")
print(f"Test examples: {len(test_dataset)}")

# Display sample images
fig, axes = plt.subplots(figsize=(10, 4), ncols=5)
for i in range(5):
    image, label = full_dataset[i]
    axes[i].imshow(image.squeeze(), cmap="gray")
    axes[i].set_title(f"Label: {label}")
    axes[i].axis("off")
plt.show()


### Part 1 Reflection

**Question:** Why do we normalize pixel intensities (mean=0.1307, std=0.3081) instead of using raw pixel values from 0 to 255?

Write your answer in the next cell.

In [ ]:
part1_reflection = ""  # TODO: Write your reflection response here


---
## Part 2: Hyperparameters & Data Splitting

### TODO 1: Choose Starter Hyperparameters

Hyperparameters are settings you choose *before* training starts.

Set reasonable starting values based on the hints in comments below.

In [ ]:
# TODO 1: Replace None with starter hyperparameter values
BATCH_SIZE = None       # Hint: Common choice between 32 and 128 (e.g., 64)
LEARNING_RATE = None   # Hint: Typical initial choice for Adam is 0.001
NUM_EPOCHS = None      # Hint: Set between 3 and 5 for fast training
HIDDEN_DIM = None      # Hint: Number of hidden units (e.g., 128)

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1


In [ ]:
if None in (BATCH_SIZE, LEARNING_RATE, NUM_EPOCHS, HIDDEN_DIM):
    raise ValueError("Please complete TODO 1 before running data loader splitting.")

train_size = int(TRAIN_RATIO * len(full_dataset))
val_size = int(VAL_RATIO * len(full_dataset))
unused_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, _ = random_split(
    full_dataset, [train_size, val_size, unused_size]
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")


--- 
## Part 3: Build a Neural Network Architecture

### Network Structure
Input ($28 \times 28 = 784$ pixels) $\rightarrow$ Linear Layer $\rightarrow$ ReLU $\rightarrow$ Linear Layer $\rightarrow$ ReLU $\rightarrow$ Output (10 classes)

### TODO 2: Fill in the layer dimensions

Complete the layer dimensions in `__init__` below.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            # TODO 2a: Fill in input dimensions (28*28) and output dimension (hidden_dim)
            nn.Linear(in_features=___, out_features=___),
            nn.ReLU(),
            # Hidden layer
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            # TODO 2b: Fill in input dimension (hidden_dim) and final classification output dimension (10)
            nn.Linear(in_features=___, out_features=___)
        )

    def forward(self, x):
        return self.network(x)

model = MLP(HIDDEN_DIM).to(device)
print(model)


---
## Part 4: Train and Validate Model

Evaluation function provided below calculates total loss and accuracy across a DataLoader.

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total


### TODO 3: Execute Training Loop

Run the training procedure. Loss functions and optimizers handle backpropagation automatically.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_accs = []
val_accs = []

print("Starting Model Training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # 1. Clear previous gradients
        optimizer.zero_grad()
        
        # 2. Forward pass: compute predictions
        outputs = model(images)
        
        # 3. Calculate Loss
        loss = criterion(outputs, labels)
        
        # 4. Backward pass: compute gradients
        loss.backward()
        
        # 5. Update model weights
        optimizer.step()

    train_loss, train_acc = evaluate(model, train_loader, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")


In [ ]:
# Plotting Learning Curves
plt.figure(figsize=(8, 5))
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning Curves")
plt.legend()
plt.grid(True)
plt.show()


---
## Part 5: Final Evaluation & Reflections

Evaluate on test data only **once** after finalizing experiments.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Final Test Accuracy: {test_acc:.4f}")


### Final Reflection Questions

Answer the following four questions in the markdown block below:

1. **Validation Set Purpose:** Why do we evaluate on a validation set during training instead of tuning hyperparameters directly on the test set?
2. **Learning Rate Impact:** What would likely happen to training if you set the learning rate extremely high (e.g., `lr = 1.0`)?
3. **Overfitting Indicators:** What signs on your training vs validation learning curves indicate that a model has started overfitting?
4. **Model Scaling:** Did increasing hidden layer dimensions (`hidden_dim`) automatically guarantee better test accuracy? Why or why not?

In [ ]:
part5_reflection = '''
1. Validation set purpose:

2. Learning rate impact:

3. Overfitting indicators:

4. Model scaling:
'''
print("Ensure reflection answers are filled in before submission.")
